In [2]:
import numpy as np
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import json


from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl

from dap_job_quality import BUCKET_NAME, PROJECT_DIR


# Load the English model with word vectors
nlp = spacy.load("en_core_web_sm")




2024-04-23 09:41:05,653 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


/Users/clare.brennan/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:

#Read in labelled data
def read_json_file(file_path):
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            data.append(json.loads(line))
    return data


#Get phrases from the labelled data
def retrieve_phrases(labelled_data:list) -> dict:
    """Retrieve phrases from spacy labelled data

    Args:
        labelled_data (list): _description_

    Returns:
        dict:{job id: [phrases labelled as benefits]}
    """    
    benefit_dict = {}
    for entry in labelled_data:
        doc = nlp(entry['text'])
        phrases_found = {'phrase': []}
        for span in entry['spans']:
            phrase = doc[span['token_start']:span['token_end']+1].text
            phrases_found['phrase'].append(phrase)
        benefit_dict[entry['id']] = phrases_found
    return benefit_dict


from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

def calculate_cosine_similarity(text:list, key_phrases:list, model=model) -> float:
    """Calculate the cosine similarity between two phrases, typically the labelled spans from benefit classifier and 
    phrases given in the sentences_for_matching dictionary (for each subcategory).

    Args:
        text (list): list of cleaned ojo sentences
        key_phrases (list): phrases to calculate cosine similarity
        model (_type_, optional): _description_. Defaults to SentenceTransformer("all-MiniLM-L6-v2")
        single_phrase (bool, optional): Transforms the vectors if only a singe element is given. Defaults to False.

    Returns:
        float: _description_
    """    
    
    embedding1 = model.encode(text)
    embedding2 = model.encode(key_phrases)
    
    
    if len(text)==1:
        embedding1 = embedding1.reshape(1,-1)
    else: pass
        
    if len(key_phrases)==1:
        embedding2 = embedding2.reshape(1,-1)
    else: pass

    return cosine_similarity(embedding1, embedding2)

In [23]:
sentences_for_matching = {'CAREER': 
    [' develop their career both technically and or into management.',
     'You will be able to develop your career with a wide variety of opportunities.',
     'learn from the best and start a successful career',
     'opportunities to develop your career',
     'advance your career in this critical role'
    ],
    'L&D': ['Fantastic training and development',
            'Learning and development is central to our engineering ethos',
            ' We have access to a wide variety of classroom  and online learning, as well as our own development programmes and schemes',
            'Financial support for learning and development',
            'We will provide you with all the tools and support you need to achieve your professional development ambitions']
    }

In [31]:
from dap_job_quality import PROJECT_DIR

labelled_data = read_json_file(f'{PROJECT_DIR}/inputs/labelled/job_sentences_labelled_20240418.jsonl')

In [29]:
labelled_phrase = retrieve_phrases(labelled_data)

In [35]:
for i in labelled_phrase:
    for j in labelled_phrase[i]['phrase']:
        for k in list(sentences_for_matching.keys()):
            print(f'Cosine similarity between {j} and {k} is {calculate_cosine_similarity(j, sentences_for_matching[k])}')

Batches: 100%|██████████| 1/1 [00:00<00:00, 86.48it/s]


ValueError: Expected 2D array, got 1D array instead:
array=[-5.86370379e-02  7.00136349e-02  5.27930679e-03  4.92692031e-02
 -1.77862234e-02  8.59453455e-02  3.89950611e-02  6.55333623e-02
 -8.63559768e-02 -3.92829031e-02 -5.81784733e-03 -1.02386296e-01
 -2.51957774e-02  1.94028541e-02  5.16674966e-02 -1.28775463e-02
  2.54007429e-03 -7.44977295e-02  8.79007578e-03 -3.54839973e-02
 -1.43222613e-02 -4.68333066e-02 -2.31570564e-02 -5.91365322e-02
  5.37846275e-02 -1.86959188e-02  6.85446262e-02  2.21546330e-02
  3.21936905e-02  2.63955630e-02 -3.90396733e-03  1.97090041e-02
  8.88094902e-02 -5.35752513e-02 -2.76814289e-02  1.76702235e-02
 -3.12551036e-02 -6.98265210e-02 -3.23010944e-02  2.81989090e-02
 -5.88541068e-02 -6.31685406e-02 -6.99709579e-02  9.05734207e-03
  2.03933306e-02 -5.30883521e-02  3.73503263e-03  1.28867161e-02
 -1.18576270e-02  3.16453613e-02  4.46947180e-02  2.89717270e-03
  6.26115408e-03 -5.52113578e-02 -5.50143570e-02  2.58567231e-03
 -7.49079138e-02 -3.19429263e-0

In [37]:
cosine_similarity(labelled_phrase[1]['phrase'], sentences_for_matching['CAREER'])

KeyError: 1

dict

In [27]:
labelled_phrase

{45238811: ['On offer is a competitive basic salary plus generous benefits package, inclusive of; 10 - 20% Annual bonus   28 crates of our products per year   Subsidised on-site café   Industry-leading pension contribution   Medical and Dental Cash Plan   Life assurance   25 days holiday with option to purchase 5 extra days   Flexible Working'],
 45243572: ['work life balance',
  'make what you do matter',
  'Work  life balance',
  'working days with alternate weekends',
  'Competitive rate of pay + travel costs',
  'profit share scheme',
  'holiday, retail and leisure discounts'],
 45255089: ['Remote working30', '£41 400 pro rata28 Hours per week.'],
 45258747: ['£20 PH + PAID BREAKS   PERMANENT',
  'excellent CQC',
  'highly trained and friendly team offering the highest standards of care.',
  'Registered Nurse Position Renumeration .',
  '£20ph.',
  'Paid Breaks.',
  'DBS Paid.',
  'Free Uniform, Training and Salary.',
  '28 Days Holiday.',
  'Excellent progression opportunities.'],

In [ ]:

career_cosine_similarity = calculate_cosine_similarity(all_text, )

Batches: 100%|██████████| 1/1 [00:00<00:00, 98.08it/s]


In [ ]:
ld_cosine_similarity = calculate_cosine_similarity(flattened_lb_list, ld_sent_master)

Batches: 100%|██████████| 1/1 [00:00<00:00, 85.29it/s]
